# 📚 Multi-PDF RAG App with Source Citations — Fresher's Guide

Welcome! This notebook builds a **Retrieval-Augmented Generation (RAG)** web app that lets you upload multiple PDFs at once, ask questions across all of them simultaneously, and get answers backed by **exact file names and page numbers**.

---
## 🧠 How RAG Works (Simple Analogy)
Imagine you have 10 textbooks and someone asks a specific question:
1. **Without RAG (Normal ChatGPT):** It answers only from general training memory and cannot read your private PDFs.
2. **With RAG (Our App):** It flips straight to the exact pages mentioning the topic, reads those paragraphs, and writes an answer citing **which file and page number** it came from.

### The 4 Core Building Blocks

| Step | Technical Term | Plain-English Meaning | Tool Used |
| :--- | :--- | :--- | :--- |
| **1** | **Chunking** | Read PDF text and slice long pages into small paragraphs (~1,000 characters). | `PyPDFLoader` & `RecursiveCharacterTextSplitter` |
| **2** | **Embeddings** | Convert each paragraph into numbers (vectors) capturing its **meaning**. | `OpenAIEmbeddings` (`text-embedding-3-small`) |
| **3** | **Vector Database** | Store all vectors in one searchable index for fast similarity search. | `FAISS` + `st.session_state` |
| **4** | **Grounded Answer** | Send top matching chunks (with **file name & page number**) to GPT. | `ChatOpenAI` (`gpt-4o-mini`) |

---
## Step 1: Install Required Libraries
We install the Python packages needed for the UI (`streamlit`), AI pipeline (`langchain`), PDF reading (`pypdf`), vector search (`faiss-cpu`), and API key loading (`python-dotenv`).

In [ ]:
!pip install -q streamlit langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv

---
## Step 2: Configure Your OpenAI API Key Safely
* Never hardcode API keys in your Python files. Store `OPENAI_API_KEY=sk-...` in a `.env` file instead.
* *(Skip running the cell below if you already added your real key to `.env` in VS Code.)*

In [ ]:
# Uncomment and run ONLY if you need to create a new .env file:
# %%writefile .env
# OPENAI_API_KEY=sk-proj-your-actual-key-here

---
## Step 3: Build the Complete Web App (`rag_app.py`)
### Key Fresher Takeaways in This Code:
* **`%%writefile rag_app.py`**: Saves the entire cell into a standalone Python file that Streamlit can run.
* **`load_dotenv(override=True)`**: Loads your `.env` key and ignores any stale keys cached in Windows.
* **`st.session_state`**: Keeps your FAISS vector database in memory when you ask multiple questions.
* **1-Indexed Pages (`page + 1`)**: `PyPDFLoader` starts counting pages at `0`, so we add `1` for human-readable page numbers.
* **Disagreement Handling**: The system prompt instructs GPT to explicitly state when two PDFs disagree rather than silently picking one.

In [ ]:
%%writefile rag_app.py
import os
import tempfile
import streamlit as st
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

# 1. Load API key from .env file (override=True ignores stale Windows keys)
load_dotenv(override=True)

st.set_page_config(page_title="Multi-PDF RAG with Sources", layout="wide")
st.title("📚 Multi-PDF RAG Assistant")

if not os.environ.get("OPENAI_API_KEY"):
    st.error("OPENAI_API_KEY not found. Please add it to your .env file.")
    st.stop()

# 2. Keep vector database in session memory across reruns
if "vector_store" not in st.session_state:
    st.session_state.vector_store = None
if "chunk_count" not in st.session_state:
    st.session_state.chunk_count = 0

# 3. Sidebar: Upload multiple PDFs & build FAISS index
with st.sidebar:
    st.header("1. Upload Documents")
    uploaded_files = st.file_uploader(
        "Upload multiple PDFs",
        type=["pdf"],
        accept_multiple_files=True
    )
    chunk_size = st.slider("Chunk Size (characters)", 500, 2000, 1000, 100)
    chunk_overlap = st.slider("Chunk Overlap", 50, 400, 150, 50)

    if st.button("Build Vector Store", type="primary") and uploaded_files:
        with st.spinner("Processing and indexing PDFs..."):
            all_docs = []

            for uploaded_file in uploaded_files:
                # Save uploaded bytes to a temp file so PyPDFLoader can read it
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
                    tmp_file.write(uploaded_file.getvalue())
                    tmp_path = tmp_file.name

                try:
                    docs = PyPDFLoader(tmp_path).load()
                    # Replace random temp filename with the real PDF filename
                    for doc in docs:
                        doc.metadata["source"] = uploaded_file.name
                    all_docs.extend(docs)
                finally:
                    os.remove(tmp_path)

            # Split documents into chunks while preserving file & page metadata
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            )
            chunks = splitter.split_documents(all_docs)

            # Embed chunks and store them in one FAISS vector database
            embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
            st.session_state.vector_store = FAISS.from_documents(chunks, embeddings)
            st.session_state.chunk_count = len(chunks)

            st.success(f"Indexing complete! **{st.session_state.chunk_count} chunks** indexed across {len(uploaded_files)} file(s).")

    if st.session_state.chunk_count > 0:
        st.info(f"Active Index: {st.session_state.chunk_count} chunks ready.")

# 4. Main Area: Ask questions & show sources
st.header("2. Ask Questions Across Your Documents")
query = st.text_input("Enter your question:")

if query:
    if st.session_state.vector_store is None:
        st.warning("Please upload PDFs and click 'Build Vector Store' in the sidebar first.")
    else:
        with st.spinner("Searching documents and generating answer..."):
            retriever = st.session_state.vector_store.as_retriever(search_kwargs={"k": 5})
            retrieved_docs = retriever.invoke(query)

            formatted_context = ""
            sources_set = set()

            for i, doc in enumerate(retrieved_docs):
                file_name = doc.metadata.get("source", "Unknown File")
                page_num = doc.metadata.get("page", 0) + 1  # Convert 0-based index to 1-based page
                sources_set.add((file_name, page_num))
                formatted_context += f"\n--- Excerpt {i+1} [File: {file_name}, Page: {page_num}] ---\n{doc.page_content}\n"

            prompt = ChatPromptTemplate.from_messages([
                ("system",
                 "You are a precise research assistant analyzing multiple documents. "
                 "Answer the user's question using ONLY the provided context excerpts. "
                 "CRITICAL INSTRUCTION: If the documents contain conflicting or contradictory information, "
                 "do NOT quietly pick one. Explicitly state that the documents disagree, and clearly present "
                 "what each source says along with its file name and page number. "
                 "If the answer is not in the context, state that you cannot find it in the uploaded files."),
                ("human", "Context:\n{context}\n\nQuestion: {question}")
            ])

            llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
            response = (prompt | llm).invoke({"context": formatted_context, "question": query})

            st.subheader("Answer")
            st.write(response.content)

            st.subheader("Sources Cited")
            for file_name, page_num in sorted(sources_set):
                st.markdown(f"- **{file_name}** — Page {page_num}")


---
## Step 4: Launch the Streamlit App
1. Run the cell below (or run `streamlit run rag_app.py` in your terminal).
2. Open **`http://localhost:8501`** in your browser.
3. Upload multiple PDFs (`Ctrl + A` in your folder), click **Build Vector Store**, check the indexed chunk count, and ask questions!

In [ ]:
!streamlit run rag_app.py

---
## 🎓 Step 5: Fresher Interview & Viva Cheat Sheet
* **Why use `chunk_overlap` (150 chars)?** So sentences aren't sliced cleanly in half at chunk boundaries, preserving context between neighboring chunks.
* **If one simple question cites 5 different pages, what is wrong?** Your `chunk_size` is likely too small (fragmenting one idea across many chunks) or `k=5` is pulling loosely related noise.
* **How does the app handle conflicting PDFs?** Every chunk is tagged `[File: name, Page: number]` before entering the prompt, and the system prompt forces the LLM to present both conflicting viewpoints with citations.